In [1]:
import os
from pathlib import Path

#from data.minbpe import BasicTokenizer as Tokenizer
from data.minbpe import RegexTokenizer as Tokenizer

import dotenv
import torch
dotenv.load_dotenv("configs/local.env")

from datasets import load_dataset

In [2]:
#### 1. dataset
repo_id = "OpenAssistant/oasst1"
ds = load_dataset(repo_id)

train_data = ds['train'].data.to_pandas()
val_data = ds['validation'].data.to_pandas()

print("all languages:", train_data['lang'].unique())

train_en = train_data[train_data['lang'] == 'en']['text']
val_en = val_data[val_data['lang'] == 'en']['text']

train_text = ' '.join(train_en.to_list()) 
val_text = ' '.join(val_en.to_list()) 

print("--> dataset: train_en={:_}, train_text={:_}, val_en={:_}, val_text={:_}".format(train_en.shape[0], len(train_text), val_en.shape[0], len(val_text)))

Using the latest cached version of the dataset since OpenAssistant/oasst1 couldn't be found on the Hugging Face Hub (offline mode is enabled).
Found the latest cached dataset configuration 'default' at data/huggingface/datasets/OpenAssistant___oasst1/default/0.0.0/fdf72ae0827c1cda404aff25b6603abec9e3399b (last modified on Fri Aug 22 13:05:21 2025).


all languages: ['en' 'es' 'de' 'ru' 'ja' 'pt-BR' 'ca' 'fr' 'pl' 'vi' 'zh' 'hu' 'ko' 'eu'
 'it' 'uk-UA' 'id' 'ar' 'fi' 'tr' 'da' 'th' 'sv' 'cs']
--> dataset: train_en=39_283, train_text=21_071_331, val_en=2_022, val_text=1_082_484


In [3]:
#### 2. tokenizer
tokenizer = Tokenizer()

tokenizer.train(train_text + " " + val_text, vocab_size=1024)

vocab = tokenizer.vocab

msg = "Hello, world!"
tokens = tokenizer.encode(msg)
decoded_msg = tokenizer.decode(tokens)
print(f"--> tokenizer: {repr(msg)} -> {tokens} -> {repr(decoded_msg)}")

max_vocab_id = list(tokenizer.vocab.keys())[-1]

tokenizer.special_tokens = {
    "<|startoftext|>": max_vocab_id + 1,
    "<|separator|>": max_vocab_id + 2,
    "<|endoftext|>": max_vocab_id + 3,
    "<|unk|>": max_vocab_id + 4,
    "<|padding|>": max_vocab_id + 5,
}

vocab_size = len(tokenizer.vocab) + len(tokenizer.special_tokens)
print(f"--> vocab_size: {vocab_size:_}")

output_dir = Path("data") / "tokenizer"
output_dir.mkdir(parents=True, exist_ok=True)

tokenizer.save(file_prefix=str(output_dir / "tokenizer"))

# tokenizer = Tokenizer()
# tokenizer.load(model_file=str(output_dir / "tokenizer.model"))

SyntaxError: invalid syntax (1001415611.py, line 22)

In [4]:
with open(output_dir / "train.txt", 'w', encoding='utf-8') as f:
    f.write(train_text)

with open(output_dir / "validation.txt", 'w', encoding='utf-8') as f:
    f.write(val_text)

train_tokens = tokenizer.encode(train_text, allowed_special="all")
val_tokens = tokenizer.encode(val_text, allowed_special="all")

print(f"--> tokens: train_tokens={len(train_tokens):_}, val_tokens={len(val_tokens):_}")

train_pt = torch.tensor(train_tokens, dtype=torch.long)
val_pt = torch.tensor(val_tokens, dtype=torch.long)

torch.save(train_pt, output_dir / 'train.tokens.pt')
torch.save(val_pt, output_dir / 'validation.tokens.pt')

--> tokens: train_tokens=8_331_538, val_tokens=429_547
